# Nuclear Shape Feature Extraction

**Author: Fabien Bertillot**

This notebook computes the single-nucleus shape features of the paper from labeled nuclear segmentation masks (Cellpose masks, one integer label per nucleus, background 0). The feature-extraction functions are in `nuclear_shape_utils.py`; this notebook only configures inputs and runs the analysis.

For each nucleus, the mask is cropped around the nucleus and:

- **perimeter, major and minor axes** are computed with PyRadiomics (`shape2D`);
- **area and convex area** are computed from the OpenCV contour polygon;
- **circularity** (`4π·area/perimeter²`), **roundness** (`4·area/(π·major²)`), **solidity**, **aspect ratio** and **compactness** are derived from these;
- **EFC ratio** is computed from normalized elliptic Fourier descriptors (25 harmonics, `spatial_efd`);
- **centroid and orientation vector** are computed from the nucleus pixel coordinates. As in the paper tables, `x_centroid` is the row index and `y_centroid` the column index.

---

## 1) Load Dependencies

In [ ]:
from pathlib import Path

import pandas as pd

from nuclear_shape_utils import (
    batch_compute_nuclear_shapes,
    compute_features_from_labeled_mask,
)

## 2) Record Package Versions

This cell is optional, but useful when documenting the environment used for a run.

In [ ]:
from watermark import watermark
%reload_ext watermark
%watermark --iversions

## 3) Configure Inputs

Set `MASK_DIR` to a folder containing labeled nuclear masks. Each nucleus should have a unique positive integer label; background should be 0. Results are written as one HDF5 feature table per input mask.

The default values reproduce the paper tables: 25 elliptic Fourier harmonics, a 5-pixel margin around each nucleus, and nuclei touching the image border kept.

In [ ]:
MASK_DIR = Path(r"../data/nuclear_shapes/masks")
SHAPE_FEATURE_DIR = Path(r"../outputs/nuclear_shapes")

MASK_GLOB = "*_mask.png"
MASK_SUFFIX = "_mask"  # Removed from the file name to give image_name
HARMONICS = 25
REMOVE_BORDER = False
OVERWRITE = False

In [ ]:
# papermill passes parameters as strings
MASK_DIR = Path(MASK_DIR)
SHAPE_FEATURE_DIR = Path(SHAPE_FEATURE_DIR)

SHAPE_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
mask_paths = sorted(MASK_DIR.glob(MASK_GLOB))
print(f"Found {len(mask_paths)} masks.")

## 4) Optional Single-Mask Preview

Run this cell to inspect the feature table for the first mask before processing the full folder (about 1–2 minutes for a TMA core with ~5,000 nuclei).

In [ ]:
if mask_paths:
    preview_df = compute_features_from_labeled_mask(
        mask_paths[0],
        harmonics=HARMONICS,
        remove_border=REMOVE_BORDER,
        mask_suffix=MASK_SUFFIX,
    )
    display(preview_df.head())
else:
    print("No masks found. Check MASK_DIR and MASK_GLOB.")

## 5) Batch Process Masks

In [ ]:
batch_compute_nuclear_shapes(
    mask_paths,
    SHAPE_FEATURE_DIR,
    harmonics=HARMONICS,
    remove_border=REMOVE_BORDER,
    mask_suffix=MASK_SUFFIX,
    overwrite=OVERWRITE,
)

print(f"Processed {len(mask_paths)} masks.")

## 6) Inspect Saved Outputs

In [ ]:
feature_files = sorted(SHAPE_FEATURE_DIR.glob("features_image=*.h5"))
print(f"Found {len(feature_files)} feature files.")

if feature_files:
    display(pd.read_hdf(feature_files[0], key="df").head())